<a href="https://colab.research.google.com/github/Rapcher1/digital_twin_atf/blob/dev/Universal_API_Components%20(2)%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Ep5 Code Lab: Free APIs & Integration
# Google Colab version

!pip -q install requests

import requests
import json
from google.colab import userdata

# 1. CONFIG
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL_NAME = "llama-3.3-70b-versatile"

# Read API key from Colab Secrets
# In Colab: click the key icon (Secrets) and create GROQ_API_KEY
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets.")

print("Connected. Ready.")

# 2. QUICK TEST — raw request
def raw_call(prompt, model=MODEL_NAME):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ]
    }

    response = requests.post(BASE_URL, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"]

print(raw_call("Say hello in one short sentence."))

Connected. Ready.
Hello.


In [ ]:
# 3. THE 5 UNIVERSAL API COMPONENTS

# 1. BASE URL — address of the model server
BASE_URL = "https://api.groq.com/openai/v1/chat/completions"

# 2. AUTH KEY — proves you are authorised
API_KEY = userdata.get("GROQ_API_KEY")

# 3. AUTH HEADER
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

# 4. MODEL STRING + 5. MESSAGES LIST
payload = {
    "model": "llama-3.3-70b-versatile",
    "messages": [
        {"role": "user", "content": "Hello"}
    ]
}

# Send and parse
response = requests.post(BASE_URL, headers=headers, json=payload)
response.raise_for_status()
data = response.json()

print(data["choices"][0]["message"]["content"])

Hello. How can I help you today?


In [ ]:
# 4. THE MESSAGES LIST IN DEPTH

full_call = requests.post(
    BASE_URL,
    headers=headers,
    json={
        "model": "llama-3.3-70b-versatile",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"}
        ]
    }
)

full_call.raise_for_status()
data = full_call.json()

print("Answer:", data["choices"][0]["message"]["content"])
print("Tokens used:", data.get("usage", {}).get("total_tokens", "Not returned"))

Answer: The capital of France is Paris.
Tokens used: 56


In [ ]:
# 5. WRAPPER CLASS

class GroqClient:
    def __init__(self, api_key, base_url=BASE_URL, default_model=MODEL_NAME):
        self.api_key = api_key
        self.base_url = base_url
        self.default_model = default_model

    def chat(self, messages, model=None, temperature=0):
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }

        payload = {
            "model": model or self.default_model,
            "messages": messages,
            "temperature": temperature
        }

        response = requests.post(self.base_url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        return data["choices"][0]["message"]["content"]

client = GroqClient(api_key=API_KEY)

In [ ]:
# 6. TESTING THE WRAPPER

# Single call
answer = client.chat([
    {"role": "user", "content": "What is 2 + 2?"}
])
print("Single call:", answer)

# Multi-turn conversation
conversation = [
    {"role": "user", "content": "My name is Alex."},
    {"role": "assistant", "content": "Hello Alex!"},
    {"role": "user", "content": "What is my name?"}
]

print()
print("Multi-turn:", client.chat(conversation))

Single call: 2 + 2 = 4.

Multi-turn: Your name is Alex.


In [ ]:
# 7. FIVE EPISODES INTO ONE PIPELINE

user_name = "Alex"

system_instruction = f"""
ROLE: You are a digital twin for {user_name}.
KNOWLEDGE: Activity data from Ep1, embeddings from Ep2, attention from Ep3, prompting strategy from Ep4.
CONSTRAINTS: Only recommend based on user data.
TONE: Direct and analytical.
"""

twin_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"}
]

response = client.chat(twin_messages)
print(response)

Analyzing your activity data from Ep1, I notice a pattern of engagement with creative tasks. Your embeddings from Ep2 indicate a strong interest in problem-solving and critical thinking. The attention data from Ep3 suggests you tend to focus on complex, challenging activities. 

Considering these insights, I recommend exploring a project that combines creative problem-solving with critical thinking. This could be a puzzle, a brain teaser, or a strategic game. The prompting strategy from Ep4 indicates you respond well to open-ended, thought-provoking questions. 

To proceed, I'll provide you with a prompt: "Design a innovative solution to a real-world problem." This should align with your interests and strengths, allowing you to leverage your creative and critical thinking skills. How would you like to approach this challenge?


In [ ]:
# 8. STRETCH: TWO-TURN CONVERSATION

two_turn_messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": "What should I do next?"},
    {"role": "assistant", "content": client.chat(twin_messages)},
    {"role": "user", "content": "Give me the top 3 priorities only."}
]

print(client.chat(two_turn_messages))

Based on your data, my top 3 priorities for you are:

1. **Engage in creative problem-solving**: Leverage your creative skills to tackle complex challenges.
2. **Develop critical thinking**: Focus on activities that require critical thinking and analysis.
3. **Attempt a challenging puzzle**: Apply your problem-solving skills to a brain teaser or strategic game.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import requests
import os

activity_names = {0: 'Sleep', 1: 'Work', 2: 'Exercise', 3: 'Social', 4: 'Leisure'}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class SimpleNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.5
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.5
        self.b2 = np.zeros((1, output_size))
        self.loss_history = []

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output, learning_rate=0.1):
        m = X.shape[0]
        dz2 = output - y
        dW2 = np.dot(self.a1.T, dz2) / m
        db2 = np.sum(dz2, axis=0, keepdims=True) / m
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * self.a1 * (1 - self.a1)
        dW1 = np.dot(X.T, dz1) / m
        db1 = np.sum(dz1, axis=0, keepdims=True) / m
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1

    def train(self, X, y, epochs=1000, learning_rate=0.1, verbose=True):
        for epoch in range(epochs):
            output = self.forward(X)
            loss = np.mean((output - y) ** 2)
            self.loss_history.append(loss)
            self.backward(X, y, output, learning_rate)
            if verbose and epoch % 200 == 0:
                print(f"  Epoch {epoch}, Loss: {loss:.4f}")

    def predict(self, X):
        return self.forward(X)


class UniversalLLMClient:
    """Generic OpenAI-compatible client — works with Groq, Together, local servers, etc."""
    def __init__(self, base_url, api_key, model="llama3.3"):
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.model = model

    def _headers(self):
        return {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def chat(self, system_instruction, user_prompt, temperature=0.3, max_tokens=300):
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt},
        ]
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        response = requests.post(
            f"{self.base_url}/chat/completions",
            headers=self._headers(),
            json=payload,
            timeout=30,
        )
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]


class DigitalTwin:
    def __init__(self, activities, days_of_week, sleep_hours, mood, weather, n_activities=5):
        self.activities = np.array(activities)
        self.days_of_week = np.array(days_of_week)
        self.sleep_hours = np.array(sleep_hours)
        self.mood = np.array(mood)
        self.weather = np.array(weather)
        self.n_activities = n_activities

        # Perceptron weights/bias (fixed, hand-set)
        self.perceptron_weights = np.array([0.3, 0.5, 0.7])
        self.perceptron_bias = -0.5

        self.nn = None
        self.nn_no_time = None

        self._build_probability_models()

    # ------------------------------------------------------
    # PROBABILITY LAYER
    # ------------------------------------------------------
    def _build_probability_models(self):
        self.freq_probs = self._calculate_probabilities(self.activities)
        self.cond_probs = self._calculate_conditional_probabilities_smoothed(self.activities)

    @staticmethod
    def _calculate_probabilities(sequence):
        counts = Counter(sequence)
        total = len(sequence)
        return {a: c / total for a, c in counts.items()}

    def _calculate_conditional_probabilities_smoothed(self, sequence, alpha=1):
        transitions = {}
        for i in range(len(sequence) - 1):
            cur, nxt = sequence[i], sequence[i + 1]
            transitions.setdefault(cur, []).append(nxt)

        smoothed = {}
        for cur, nxt_list in transitions.items():
            counts = Counter(nxt_list)
            total = len(nxt_list) + alpha * self.n_activities
            smoothed[cur] = {
                a: (counts.get(a, 0) + alpha) / total
                for a in range(self.n_activities)
            }
        return smoothed

    def joint_probability(self, activity, is_weekend):
        matches = sum(1 for a, d in zip(self.activities, self.days_of_week)
                      if a == activity and (d >= 5) == is_weekend)
        return matches / len(self.activities)

    def bayes_weekend_given_activity(self, activity):
        is_weekend = self.days_of_week >= 5
        p_weekend = np.mean(is_weekend)
        weekend_acts = self.activities[is_weekend]
        p_act_given_weekend = np.mean(weekend_acts == activity) if len(weekend_acts) else 0
        p_activity = np.mean(self.activities == activity)
        return 0 if p_activity == 0 else (p_act_given_weekend * p_weekend) / p_activity

    def predict_next_distribution(self, current):
        if current not in self.cond_probs:
            return None
        return dict(sorted(self.cond_probs[current].items(), key=lambda x: -x[1]))

    def predict_next_markov(self, current):
        dist = self.predict_next_distribution(current)
        return None if dist is None else max(dist, key=dist.get)

    def prediction_entropy(self, current):
        dist = self.predict_next_distribution(current)
        if dist is None:
            return None
        probs = np.array(list(dist.values()))
        probs = probs[probs > 0]
        return -np.sum(probs * np.log2(probs))

    def sequence_probability(self, seq):
        prob = 1.0
        for i in range(len(seq) - 1):
            cur, nxt = seq[i], seq[i + 1]
            if cur not in self.cond_probs or nxt not in self.cond_probs[cur]:
                return 0.0
            prob *= self.cond_probs[cur][nxt]
        return prob

    def sample_next_activity(self, current, rng):
        if current not in self.cond_probs:
            return None
        options = list(self.cond_probs[current].keys())
        weights = list(self.cond_probs[current].values())
        return rng.choice(options, p=weights)

    def simulate_greedy(self, start, days=7):
        seq = [start]
        for _ in range(days):
            nxt = self.predict_next_markov(seq[-1])
            if nxt is None:
                break
            seq.append(nxt)
        return seq

    def simulate_sampled(self, start, days=7, seed=42):
        rng = np.random.default_rng(seed)
        seq = [start]
        for _ in range(days):
            nxt = self.sample_next_activity(seq[-1], rng)
            if nxt is None:
                break
            seq.append(nxt)
        return seq

    # ------------------------------------------------------
    # PERCEPTRON LAYER
    # ------------------------------------------------------
    def perceptron_predict(self, sleep, free_time, energy):
        inputs = np.array([sleep, free_time, energy])
        return sigmoid(np.dot(inputs, self.perceptron_weights) + self.perceptron_bias)

    # ------------------------------------------------------
    # NEURAL NETWORK LAYER
    # ------------------------------------------------------
    def _build_features_and_targets(self, include_time=True):
        X, y = [], []
        for i in range(len(self.activities) - 1):
            feats = [self.activities[i] / (self.n_activities - 1),
                     self.sleep_hours[i] / 10, self.mood[i], self.weather[i]]
            if include_time:
                feats.insert(1, self.days_of_week[i] / 7)
            X.append(feats)
            target = np.zeros(self.n_activities)
            target[self.activities[i + 1]] = 1
            y.append(target)
        return np.array(X), np.array(y)

    def train_nn(self, include_time=True, hidden_size=8, epochs=1000, learning_rate=0.3, verbose=True):
        X, y = self._build_features_and_targets(include_time=include_time)
        split = int(len(X) * 0.8)
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        model = SimpleNeuralNetwork(input_size=X.shape[1], hidden_size=hidden_size,
                                     output_size=self.n_activities)
        model.train(X_train, y_train, epochs=epochs, learning_rate=learning_rate, verbose=verbose)

        preds = np.argmax(model.predict(X_test), axis=1)
        actual = np.argmax(y_test, axis=1)
        accuracy = np.mean(preds == actual)

        if include_time:
            self.nn = model
        else:
            self.nn_no_time = model

        return model, accuracy, preds, actual

    # ------------------------------------------------------
    # PROMPTING LAYER
    # ------------------------------------------------------
    def generate_zero_shot_prompt(self, current_activity):
        predicted = self.predict_next_markov(current_activity)
        return {
            'technique': 'zero_shot',
            'task': 'activity_prediction',
            'prompt': f"I just finished {activity_names[current_activity]}. "
                      f"What's the most likely next activity?",
            'expected': activity_names[predicted] if predicted is not None else 'unknown'
        }

    def generate_few_shot_prompt(self, current_activity, n_examples=3):
        examples = []
        for i in range(len(self.activities) - 1):
            if self.activities[i] == current_activity:
                examples.append((self.activities[i], self.activities[i + 1]))
            if len(examples) >= n_examples:
                break

        prompt_lines = ["Here are observed activity transitions:"]
        for cur, nxt in examples:
            prompt_lines.append(f"- After {activity_names[cur]}, the next activity was {activity_names[nxt]}.")
        prompt_lines.append(f"Based on this pattern, what comes after {activity_names[current_activity]}?")

        return {
            'technique': 'few_shot',
            'task': 'activity_prediction',
            'examples': [{'current': activity_names[c], 'next': activity_names[n]} for c, n in examples],
            'prompt': "\n".join(prompt_lines),
            'expected': activity_names[self.predict_next_markov(current_activity)]
        }

    def generate_cot_prompt(self, day_index):
        sleep = self.sleep_hours[day_index]
        mood_v = self.mood[day_index]
        weather_v = self.weather[day_index]
        is_weekend = self.days_of_week[day_index] >= 5

        steps = [
            f"1. Sleep was {sleep:.1f} hours -> {'well-rested' if sleep >= 7 else 'low energy'}.",
            f"2. Mood score was {mood_v:.2f} -> {'positive' if mood_v >= 0.5 else 'low'}.",
            f"3. Weather score was {weather_v:.2f} -> {'good' if weather_v >= 0.5 else 'poor'}.",
            f"4. Day type: {'weekend' if is_weekend else 'weekday'}.",
        ]

        p_exercise = self.perceptron_predict(sleep, 2 if is_weekend else 1, mood_v)
        conclusion = "exercise" if p_exercise > 0.5 else "a lower-energy activity (sleep/leisure)"
        steps.append(f"5. Combining these factors, exercise probability = {p_exercise:.2%} -> "
                      f"most likely outcome: {conclusion}.")

        prompt = (
            f"Given sleep={sleep:.1f}h, mood={mood_v:.2f}, weather={weather_v:.2f}, "
            f"{'weekend' if is_weekend else 'weekday'}: think step-by-step about the "
            f"most likely activity."
        )

        return {
            'technique': 'chain_of_thought',
            'task': 'activity_prediction',
            'prompt': prompt,
            'reasoning_steps': steps,
            'expected': conclusion
        }

    def run_prompting_demo(self, current_activity=1, day_index=0):
        print("=== DIGITAL TWIN PROMPTING TECHNIQUES ===")

        zs = self.generate_zero_shot_prompt(current_activity)
        print(f"\n1. Technique: {zs['technique'].upper()}")
        print(f"Prompt: {zs['prompt']}")
        print(f"Expected: {zs['expected']}")

        fs = self.generate_few_shot_prompt(current_activity)
        print(f"\n2. Technique: {fs['technique'].upper()}")
        for i, ex in enumerate(fs['examples'], 1):
            print(f"  --- Example {i}: {ex['current']} -> {ex['next']}")
        print(f"Prompt:\n{fs['prompt']}")
        print(f"Expected: {fs['expected']}")

        cot = self.generate_cot_prompt(day_index)
        print(f"\n3. Technique: {cot['technique'].upper()}")
        print(f"Prompt: {cot['prompt']}")
        print("Steps:")
        for step in cot['reasoning_steps']:
            print(f"  {step}")
        print(f"Expected Final Answer: {cot['expected']}")

    # ------------------------------------------------------
    # SYSTEM INSTRUCTION LAYER
    # ------------------------------------------------------
    def build_system_instruction(self, n_recent_days=10):
        """
        Summarizes observed daily patterns (learning, coding, writing, revising,
        exercise, social, etc. mapped onto the activity codes) into rules an
        LLM can follow when predicting the twin's next activity.
        """
        lines = [
            "You are a digital twin of a user, predicting their next likely activity "
            "based on observed daily patterns. Follow these rules, derived from real data:",
            ""
        ]

        lines.append("Overall activity distribution:")
        for a, p in sorted(self.freq_probs.items()):
            lines.append(f"  - {activity_names[a]}: {p:.1%} of days")
        lines.append("")

        lines.append("Observed transition rules (after X, the user usually does Y):")
        for cur, dist in self.cond_probs.items():
            top_next = max(dist, key=dist.get)
            lines.append(f"  - After {activity_names[cur]} -> most likely {activity_names[top_next]} "
                         f"({dist[top_next]:.1%})")
        lines.append("")

        recent = self.activities[-n_recent_days:]
        recent_str = " -> ".join(activity_names[a] for a in recent)
        lines.append(f"Most recent {len(recent)} days, in order: {recent_str}")
        lines.append("")

        lines.append("When asked to predict the next activity, reason from these rules "
                      "and the recent sequence, not from general assumptions about people.")

        return "\n".join(lines)

    def ask_llm(self, llm_client, current_activity, n_recent_days=10):
        system_instruction = self.build_system_instruction(n_recent_days=n_recent_days)
        user_prompt = (f"I just finished {activity_names[current_activity]}. "
                       f"What's the most likely next activity, and why?")
        return llm_client.chat(system_instruction, user_prompt)


# ============================================================
# 1. COLLECT DATA
# ============================================================
np.random.seed(1)
n_days = 60
days_of_week = np.array([i % 7 for i in range(n_days)])
sleep_hours = np.random.normal(7, 1, n_days).clip(4, 10)
mood = np.random.rand(n_days)
weather = np.random.rand(n_days)

activities = []
for d in days_of_week:
    if d >= 5:
        activities.append(np.random.choice([2, 3, 4], p=[0.3, 0.4, 0.3]))
    else:
        activities.append(np.random.choice([1, 0, 2], p=[0.7, 0.2, 0.1]))
activities = np.array(activities)

twin = DigitalTwin(activities, days_of_week, sleep_hours, mood, weather)

# ============================================================
# 2. PROBABILITY FEATURES
# ============================================================
print("Overall activity probabilities:")
for a, p in sorted(twin.freq_probs.items()):
    print(f"  {activity_names[a]}: {p:.2%}")

print("\nJoint P(Exercise AND Weekend):", f"{twin.joint_probability(2, True):.2%}")
print("Bayes P(Weekend | Exercise):", f"{twin.bayes_weekend_given_activity(2):.2%}")

print("\nFull next-activity distribution after Work:")
for a, p in twin.predict_next_distribution(1).items():
    print(f"  {activity_names[a]}: {p:.2%}")

print(f"\nPrediction entropy after Work: {twin.prediction_entropy(1):.3f}")
print("Greedy 7-day simulation from Work:",
      [activity_names[a] for a in twin.simulate_greedy(1, 7)])
print("Sampled 7-day simulation from Work:",
      [activity_names[a] for a in twin.simulate_sampled(1, 7)])
print("P(Work -> Exercise -> Social):",
      f"{twin.sequence_probability([1, 2, 3]):.4%}")

# ============================================================
# 3. PERCEPTRON
# ============================================================
print("\nPerceptron P(exercise | sleep=7, free_time=2, energy=0.8):",
      f"{twin.perceptron_predict(7, 2, 0.8):.2%}")

# ============================================================
# 4. TRAIN + EVALUATE NEURAL NETWORK
# ============================================================
print("\nTraining NN with day-of-week...")
model_time, acc_time, preds_time, actual_time = twin.train_nn(include_time=True)
print(f"Accuracy with day-of-week: {acc_time:.2%}")

print("\nTraining NN without day-of-week...")
model_no_time, acc_no_time, preds_nt, actual_nt = twin.train_nn(include_time=False)
print(f"Accuracy without day-of-week: {acc_no_time:.2%}")

# ============================================================
# 5. VISUALIZE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

acts = [activity_names[k] for k in sorted(twin.freq_probs.keys())]
probs = [twin.freq_probs[k] for k in sorted(twin.freq_probs.keys())]
axes[0, 0].bar(acts, probs, color='steelblue')
axes[0, 0].set_title("Overall Activity Probabilities")

axes[0, 1].plot(model_time.loss_history, label="With day-of-week")
axes[0, 1].plot(model_no_time.loss_history, label="Without day-of-week")
axes[0, 1].set_title("Training Loss")
axes[0, 1].legend()

per_class_acc = [np.mean(preds_time[actual_time == c] == c) if np.any(actual_time == c) else 0
                  for c in range(5)]
axes[1, 0].bar([activity_names[c] for c in range(5)], per_class_acc, color='seagreen')
axes[1, 0].set_title("Per-Activity Accuracy (NN)")
axes[1, 0].tick_params(axis='x', rotation=45)

axes[1, 1].bar(["With day-of-week", "Without"], [acc_time, acc_no_time],
               color=['steelblue', 'salmon'])
axes[1, 1].set_title("Effect of Temporal Features")

plt.tight_layout()
plt.show()

# ============================================================
# 6. PROMPTING TECHNIQUES
# ============================================================
twin.run_prompting_demo(current_activity=1, day_index=5)

# ============================================================
# 7. SYSTEM INSTRUCTION + LLM INTEGRATION
# ============================================================
print("\n=== SYSTEM INSTRUCTION BUILT FROM TWIN DATA ===")
print(twin.build_system_instruction())

# -------- PUT YOUR API CREDENTIALS HERE --------
LLM_BASE_URL = "https://api.groq.com/openai/v1/chat/completions"
LLM_API_KEY = "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"
LLM_MODEL = "llama-3.3-70b-versatile"                             # <-- exact model string from provider
# ------------------------------------------------

llm_client = UniversalLLMClient(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    model=LLM_MODEL,
)


 response = twin.ask_llm(llm_client, current_activity=1)
 print("\nLLM response:\n", response)